# CampusClaim - Theme 3B: Receipt Information Extraction

Fine-tune **Gemma 4 E2B** with LoRA on ICDAR 2019 SROIE receipts, compare the plain and fine-tuned model on the same held-out receipts, and demo a receipt-to-claim app.

**Run order:** Runtime -> Change runtime type -> **T4 GPU**, then run the install and model cells, then the dataset cell and the one-receipt smoke test. Only start the 100-receipt baseline and training after the smoke test prints JSON. The install cell takes 2-4 minutes.

**What is different from the starter kit:** the starter's `evaluate()` is text-only, so this notebook replaces it with image generation + per-field scoring. The vision snippets in the starter's last section are guidance, not runnable cells; this notebook turns them into a working pipeline.

**Budget on a free T4:** install ~3 min, model download ~5 min, dataset download ~2 min, baseline on 100 receipts ~3-5 min, 120 training steps ~20-30 min, fine-tuned eval ~3-5 min. Gemma 4 runs in float32 on a T4 because float16 gives NaN gradients; that is expected and is why the image side is capped at 1024 px.

In [ ]:
%%capture
# Installs Unsloth and compatible library versions (2-4 minutes, output hidden).
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU found. Select Runtime -> Change runtime type -> T4 GPU, then rerun from the top.")

## Step 1: Load the vision model

`FastVisionModel` returns a **processor** (not a tokenizer). The Gemma 4 chat template is applied to the processor.

In [ ]:
import gc
import torch
from unsloth import FastVisionModel, get_chat_template

MODEL_NAME = "unsloth/gemma-4-E2B-it"   # E4B is more capable but slower and tighter on a T4
INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."

# Unsloth forces float32 for Gemma 4 because float16 produces NaN gradients.
# That is expected on a T4. Do not fight it; keep memory down with an encoded
# image column and a smaller image side instead.
model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit = True,                  # keeps the model inside a free T4's 16 GB
    use_gradient_checkpointing = "unsloth",
)
processor = get_chat_template(processor, "gemma-4")
gc.collect()
torch.cuda.empty_cache()
print("Loaded", MODEL_NAME)
print("model device:", next(model.parameters()).device)
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated, {torch.cuda.memory_reserved()/1e9:.1f} GB reserved")


## Step 2: Load the receipts

SROIE fields: `entities.company`, `entities.date`, `entities.address`, `entities.total`. Train on the 625 usable training receipts (one of the 626 has an empty total), evaluate on a fixed random sample of the 361 test receipts. The test receipts are never trained on.

In [ ]:
import gc
from datasets import load_dataset

FIELDS = ["company", "date", "address", "total"]

train_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="train")
test_data  = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")

def usable_entities(entities):
    entities = entities or {}
    return all(str(entities.get(k, "")).strip() for k in FIELDS)

# Filter on metadata only. Iterating the full image columns decodes every
# receipt into RAM; that is what crashed the 12.7 GB Colab runtime.
train_meta = train_data.select_columns(["key", "entities"])
test_meta  = test_data.select_columns(["key", "entities"])

train_indices = [i for i, row in enumerate(train_meta) if usable_entities(row["entities"])]
test_indices  = [i for i, row in enumerate(test_meta) if usable_entities(row["entities"])]

# Frozen with scripts/freeze_eval_sample.py (seed 0). Do not resample.
EVAL_KEYS = [
    "X51006438346",
    "X51006556740",
    "X51005433518",
    "X51005806692",
    "X51006619869",
    "X51006619700",
    "X51006556610",
    "X51006329388",
    "X51006619564",
    "X51006414425",
    "X51007231346",
    "X51005745213",
    "X51006619790",
    "X51005677327",
    "X51006008105",
    "X51008042780",
    "X51005568895",
    "X51007433809",
    "X51005763999",
    "X51006828201",
    "X51007339127",
    "X510056849111",
    "X51006334742",
    "X51005582745",
    "X51005447844",
    "X51006387737",
    "X51006619506",
    "X51007103597",
    "X51005587261",
    "X51006401940",
    "X51006556826",
    "X51006334819",
    "X51007339161",
    "X51007846303",
    "X51005724623",
    "X51006913032",
    "X51008099086",
    "X51006556865",
    "X51006647933",
    "X51005806695",
    "X51005444040",
    "X51006912998",
    "X51005230657",
    "X51005568894",
    "X51006555833",
    "X00016469670",
    "X51007339163",
    "X51006619760",
    "X51006387931",
    "X51005757308",
    "X51006349085",
    "X51005444041",
    "X51005719898",
    "X51007103687",
    "X51005745298",
    "X51005757282",
    "X51005677333",
    "X51006912959",
    "X51006557173",
    "X51005568892",
    "X51005568866",
    "X51006335818",
    "X51006619863",
    "X51006619704",
    "X51005621482",
    "X51006329183",
    "X51007846289",
    "X51006327953",
    "X51005675103",
    "X51007579725",
    "X51007846304",
    "X51006857137",
    "X51007846290",
    "X51007225406",
    "X51006248253",
    "X51006557115",
    "X51007339105",
    "X51009568881",
    "X51006334927",
    "X51005757292",
    "X51006311714",
    "X51005719888",
    "X51005719895",
    "X51005719889",
    "X51005361908",
    "X51007846268",
    "X51006619509",
    "X51005447841",
    "X51005568890",
    "X51005676534",
    "X51005684949",
    "X51005365187",
    "X51007231372",
    "X51006502535",
    "X51006008090",
    "X51005757220",
    "X51005742110",
    "X51006556734",
    "X51006008083",
    "X51006557187",
]

eval_keys = {test_meta[i]["key"] for i in test_indices}
eval_indices = [i for i in test_indices if test_meta[i]["key"] in set(EVAL_KEYS)]
missing = sorted(set(EVAL_KEYS) - eval_keys)
assert not missing, f"frozen keys missing from the dataset: {missing[:5]}"
assert len(eval_indices) == len(EVAL_KEYS), f"expected {len(EVAL_KEYS)} eval rows, got {len(eval_indices)}"

class LazyRows:
    """Index a HF dataset and decode one receipt image at a time."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        return self.dataset[self.indices[index]]

eval_rows = LazyRows(test_data, eval_indices)

print(f"train usable: {len(train_indices)}/{len(train_data)}  test usable: {len(test_indices)}/{len(test_data)}  evaluating on: {len(eval_rows)}")
print("example entities:", {k: train_meta[train_indices[0]]["entities"][k] for k in FIELDS})

# Metadata views are no longer needed; drop them so Colab's 12.7 GB stays clear.
del train_meta, test_meta
gc.collect()


In [ ]:
import gc, json
from PIL import Image
from datasets import Dataset

MAX_IMAGE_SIDE = 1024   # 1600 px plus Gemma 4's forced float32 did not fit in 12.7 GB RAM

def prepare_image(image):
    image = image.convert("RGB")
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    return image

def to_conversation(sample):
    target = json.dumps({k: sample["entities"][k] for k in FIELDS}, ensure_ascii=False)
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": INSTRUCTION},
                {"type": "image"},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]},
        ],
        # Keep the image in its own column. datasets stores it as encoded bytes,
        # so the collator only decodes the current batch instead of all 625 receipts.
        "images": [prepare_image(sample["image"])],
    }

train_examples = [to_conversation(train_data[i]) for i in train_indices]
train_conversations = Dataset.from_list(train_examples)
del train_examples
gc.collect()

print("training conversations:", len(train_conversations))
print(train_conversations[0]["messages"][1]["content"][0]["text"])


## Step 3: Baseline - the plain model

Generation takes an image + the same instruction used in training. Scoring is per field, after normalisation, so a formatting difference is not counted as a wrong extraction.

In [ ]:
import re, json, time
from datetime import datetime

def parse_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        return None

MONTHS = {m: i for i, m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], 1)}

def norm_text(value):
    s = re.sub(r"[^a-z0-9]+", " ", str(value or "").lower())
    return re.sub(r"\s+", " ", s).strip()

def norm_date(value):
    s = str(value or "").strip().lower()
    m = re.match(r"^(\d{1,2})[\s/-]+([a-z]{3})[a-z]*[\s/-]+(\d{2,4})$", s)
    if m:
        d, mon, y = m.groups()
        y = ("20" + y) if len(y) == 2 else y
        return f"{int(d):02d}/{MONTHS[mon]:02d}/{y}"
    digits = re.findall(r"\d+", s)
    if len(digits) == 3:
        d, mo, y = digits
        y = ("20" + y) if len(y) == 2 else y
        return f"{int(d):02d}/{int(mo):02d}/{y}"
    return "/".join(digits)

def norm_total(value):
    m = re.search(r"-?\d+(?:\.\d+)?", str(value or "").replace(",", ""))
    return float(m.group(0)) if m else None

def token_f1(pred, gold):
    p, g = norm_text(pred).split(), norm_text(gold).split()
    if not p and not g: return 1.0
    if not p or not g: return 0.0
    common = sum((__import__("collections").Counter(p) & __import__("collections").Counter(g)).values())
    if common == 0: return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

def score(prediction, expected):
    pred = parse_json(prediction)
    gold = {k: expected["entities"][k] for k in FIELDS}
    if pred is None:
        return {"valid_json": 0, "company": 0, "date": 0, "address": 0, "total": 0, "address_f1": 0.0}
    return {
        "valid_json": 1,
        "company": int(norm_text(pred.get("company")) == norm_text(gold["company"])),
        "date": int(norm_date(pred.get("date")) == norm_date(gold["date"])),
        "address": int(norm_text(pred.get("address")) == norm_text(gold["address"])),
        "total": int(norm_total(pred.get("total")) is not None and norm_total(gold["total"]) is not None
                     and abs(norm_total(pred.get("total")) - norm_total(gold["total"])) <= 0.01),
        "address_f1": token_f1(pred.get("address"), gold["address"]),
    }

def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=128):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(prepare_image(image), input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate_vision(rows, label, max_new_tokens=128):
    model.eval()
    records = []
    started = time.time()
    for i, sample in enumerate(rows, start=1):
        prediction = generate_from_image(sample["image"], max_new_tokens=max_new_tokens)
        records.append({
            "key": sample["key"],
            "expected": json.dumps({k: sample["entities"][k] for k in FIELDS}, ensure_ascii=False),
            label: prediction,
            **score(prediction, sample),
        })
        if i % 10 == 0 or i == len(rows):
            print(f"\r{label}: {i}/{len(rows)}  ({time.time() - started:.0f}s)", end="")
    print()
    return records

import pandas as pd

def metrics(records):
    df = pd.DataFrame(records)
    cols = ["valid_json", "company", "date", "address", "total", "address_f1"]
    out = df[cols].mean().mul(100).round(1)
    out.index = ["valid JSON", "company exact", "date exact", "address exact", "total exact", "address token-F1"]
    return df, out

### Smoke test before the 100-receipt run

One generation on the first frozen receipt. If this prints JSON, the full run will work; if it errors, fix it before spending GPU time.

In [ ]:
smoke = eval_rows[0]
print("key:", smoke["key"])
print("expected:", {k: smoke["entities"][k] for k in FIELDS})
print("prediction:", generate_from_image(smoke["image"]))

In [ ]:
baseline_records = evaluate_vision(eval_rows, "plain_model")
baseline_df, baseline_metrics = metrics(baseline_records)
print("\nPlain Gemma 4 E2B (%):")
print(baseline_metrics.to_string())
baseline_df[["key", "expected", "plain_model", "valid_json", "total"]].head(5)

## Step 4: Fine-tune with LoRA

Vision layers are trained too, because the task depends on reading the receipt, not only on formatting the answer. `r=16` is a reasonable starting point; the starter's text-only settings do not apply here.

In [ ]:
LORA_RANK = 16
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_RANK,
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

TRAIN_STEPS = 120      # about 0.75 epoch over the 625 usable training receipts
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4
MAX_LENGTH = 2048

import gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")

trainer = SFTTrainer(
    model = model,
    train_dataset = train_conversations,
    processing_class = processor,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        max_steps = TRAIN_STEPS,
        learning_rate = LEARNING_RATE,
        logging_steps = 5,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = MAX_LENGTH,
        # Decode images in the main process; worker copies would duplicate the
        # Arrow image column and defeat the memory fix.
        dataloader_num_workers = 0,
    ),
)

trainer_stats = trainer.train()
print(f"Training finished in {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes.")


## Step 5: Evaluate the fine-tuned model on the same receipts

In [ ]:
tuned_records = evaluate_vision(eval_rows, "fine_tuned_model")
tuned_df, tuned_metrics = metrics(tuned_records)

comparison = pd.DataFrame({
    "Plain Gemma (%)": baseline_metrics,
    "Fine-tuned Gemma (%)": tuned_metrics,
})
comparison["Change (points)"] = (comparison["Fine-tuned Gemma (%)"] - comparison["Plain Gemma (%)"]).round(1)
print(comparison.to_string())

In [ ]:
# Failure examples: the receipts the fine-tuned model still gets wrong.
merged = baseline_df[["key", "expected", "plain_model"]].merge(
    tuned_df[["key", "fine_tuned_model", "company", "date", "address", "total"]], on="key"
)
failures = merged[(merged[["company", "date", "address", "total"]].sum(axis=1) < 4)]
print(f"{len(failures)} of {len(merged)} receipts had at least one wrong field.")
failures.head(8)

In [ ]:
# Save everything M2 needs. Download the results/ folder from the Colab file browser.
from pathlib import Path

out = Path("results")
out.mkdir(exist_ok=True)

baseline_metrics.to_json(out / "baseline_metrics.json", indent=2)
tuned_metrics.to_json(out / "tuned_metrics.json", indent=2)
comparison.to_csv(out / "comparison.csv")
failures.to_csv(out / "failures.csv", index=False)
baseline_df.to_csv(out / "baseline_predictions.csv", index=False)
tuned_df.to_csv(out / "tuned_predictions.csv", index=False)

settings = {
    "model": MODEL_NAME,
    "instruction": INSTRUCTION,
    "eval_keys": len(EVAL_KEYS),
    "eval_seed": 0,
    "image_max_side": MAX_IMAGE_SIDE,
    "lora_rank": LORA_RANK,
    "train_steps": TRAIN_STEPS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "max_length": MAX_LENGTH,
}
(out / "settings.json").write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))
print("wrote:", ", ".join(sorted(p.name for p in out.iterdir())))

## Step 6: The application

Upload a receipt, see the plain and fine-tuned extractions side by side, correct the fields, and copy the JSON. The fine-tuned model is the default; the plain model runs with the LoRA adapters temporarily switched off.

In [ ]:
import gradio as gr

def extract_both(image):
    if image is None:
        raise gr.Error("Upload a receipt image first.")
    model.eval()
    with model.disable_adapter():
        plain = generate_from_image(image)
    tuned = generate_from_image(image)
    data = parse_json(tuned) or {}
    return plain, tuned, str(data.get("company", "")), str(data.get("date", "")), str(data.get("address", "")), str(data.get("total", ""))

def build_summary(company, date, address, total, purpose):
    rows = {
        "merchant": company.strip() or "(missing)",
        "date": date.strip() or "(missing)",
        "address": address.strip() or "(missing)",
        "total": total.strip() or "(missing)",
        "purpose": purpose.strip() or "(missing)",
    }
    missing = [k for k, v in rows.items() if v == "(missing)"]
    lines = ["**Claim draft**", ""]
    lines += [f"- {k.title()}: {v}" for k, v in rows.items()]
    lines += ["", f"Missing: {', '.join(missing) if missing else 'nothing'}"]
    lines += ["", "Approval stays with the college's authorised reviewer."]
    return "\n".join(lines)

with gr.Blocks(title="CampusClaim - receipt to claim") as demo:
    gr.Markdown("# CampusClaim\nExtract receipt fields with Gemma 4, then prepare a claim draft.")
    with gr.Row():
        image_in = gr.Image(type="pil", label="Receipt image")
        with gr.Column():
            plain_out = gr.Textbox(label="Plain Gemma (baseline)", lines=5)
            tuned_out = gr.Textbox(label="Fine-tuned Gemma", lines=5)
    extract_btn = gr.Button("Extract", variant="primary")
    with gr.Row():
        company = gr.Textbox(label="Company")
        date = gr.Textbox(label="Date")
        address = gr.Textbox(label="Address")
        total = gr.Textbox(label="Total")
    purpose = gr.Textbox(label="Event / purchase purpose", placeholder="e.g. printing flyers for the robotics club")
    summary_btn = gr.Button("Build claim draft")
    summary_out = gr.Markdown()
    extract_btn.click(extract_both, inputs=image_in, outputs=[plain_out, tuned_out, company, date, address, total])
    summary_btn.click(build_summary, inputs=[company, date, address, total, purpose], outputs=summary_out)
demo.launch(share=True)

## Step 7: Save the adapters before the session ends

Free Colab disconnects. Download the zip, or push to Hugging Face if you have a token.

In [ ]:
model.save_pretrained("campusclaim_gemma4_lora")
processor.save_pretrained("campusclaim_gemma4_lora")
!zip -qr campusclaim_gemma4_lora.zip campusclaim_gemma4_lora
print("Saved campusclaim_gemma4_lora.zip - download it from the folder icon.")
# model.push_to_hub("your-username/campusclaim-gemma4-lora", token="YOUR_HF_TOKEN")

## Optional: the 3B "combined approach" bonus (policy RAG)

If there is time, add a second tab that answers reimbursement questions from **real** college finance/activity documents, with document + page citations and a refusal when the documents do not cover the question. Keep it separate from the extraction benchmark, and do not let the model invent a policy. The Theme 2 starter kit has a working RAG pipeline you can copy into a second notebook rather than mixing it into this GPU notebook.

## Honest reporting notes

- SROIE is Malaysian receipts (RM, dd/mm/yyyy). A campus receipt from your college is a **domain shift**; report a small local test separately and do not mix it into the SROIE numbers.
- Report `valid JSON` separately from field accuracy: a model that emits prose scores 0 on all fields, which is a real failure mode.
- Exact address match is strict. Report token-F1 alongside it so a missing comma is not presented as a total failure.
- State the evaluation subset size and seed. Do not compare a 100-receipt baseline with a 361-receipt fine-tuned run.
- If fine-tuning does not beat the baseline, say so and show the failures; the problem statement explicitly asks where it helped and where it did not.